# 02 — Data Cleaning
Cleans, deduplicates, and gap-fills the raw datasets, then loads them into the SQLite database. This mirrors the logic in `scripts/compute_metrics.py` — see that script for the production/CLI version.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
DB_PATH = Path("../data/db/bluestock_mf.db")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

fund_master  = pd.read_csv(RAW_DIR / "01_fund_master.csv")
nav_history  = pd.read_csv(RAW_DIR / "02_nav_history.csv", parse_dates=["date"])
scheme_perf  = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
transactions = pd.read_csv(RAW_DIR / "08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings     = pd.read_csv(RAW_DIR / "09_portfolio_holdings.csv")
benchmark    = pd.read_csv(RAW_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

print("Raw shapes:", {k: v.shape for k, v in {
    "fund_master": fund_master, "nav_history": nav_history,
    "scheme_performance": scheme_perf, "transactions": transactions,
    "holdings": holdings, "benchmark": benchmark
}.items()})

Raw shapes: {'fund_master': (40, 15), 'nav_history': (46000, 3), 'scheme_performance': (40, 19), 'transactions': (32778, 13), 'holdings': (322, 8), 'benchmark': (8050, 3)}


## Deduplicate
Drop exact duplicate rows that can occur from repeated ingestion runs.

In [2]:
before = {
    "fund_master": len(fund_master), "nav_history": len(nav_history),
    "scheme_performance": len(scheme_perf), "transactions": len(transactions),
    "holdings": len(holdings), "benchmark": len(benchmark),
}

fund_master  = fund_master.drop_duplicates()
nav_history  = nav_history.drop_duplicates(subset=["amfi_code", "date"])
scheme_perf  = scheme_perf.drop_duplicates(subset=["amfi_code"])
transactions = transactions.drop_duplicates()
holdings     = holdings.drop_duplicates()
benchmark    = benchmark.drop_duplicates(subset=["date"])

after = {
    "fund_master": len(fund_master), "nav_history": len(nav_history),
    "scheme_performance": len(scheme_perf), "transactions": len(transactions),
    "holdings": len(holdings), "benchmark": len(benchmark),
}

for k in before:
    print(f"{k:<20} {before[k]:>7} -> {after[k]:>7}  ({before[k]-after[k]} duplicates removed)")

fund_master               40 ->      40  (0 duplicates removed)
nav_history            46000 ->   46000  (0 duplicates removed)
scheme_performance        40 ->      40  (0 duplicates removed)
transactions           32778 ->   32778  (0 duplicates removed)
holdings                 322 ->     322  (0 duplicates removed)
benchmark               8050 ->    1150  (6900 duplicates removed)


## Fill NAV gaps on weekends/holidays
Reindex each scheme's NAV series to the full calendar date range, then forward-fill so weekends/holidays inherit the prior trading day's NAV — avoiding artificial gaps in return calculations downstream.

In [3]:
filled_frames = []
for code_, grp in nav_history.groupby("amfi_code"):
    grp = grp.set_index("date").sort_index()
    full_range = pd.date_range(grp.index.min(), grp.index.max(), freq="D")
    grp = grp.reindex(full_range)
    grp["amfi_code"] = code_
    grp["nav"] = grp["nav"].ffill()
    grp.index.name = "date"
    filled_frames.append(grp.reset_index())

nav_history_clean = pd.concat(filled_frames, ignore_index=True)
print(f"nav_history: {len(nav_history)} -> {len(nav_history_clean)} rows after gap-filling")
print(f"Remaining nulls: {nav_history_clean['nav'].isnull().sum()}")

nav_history: 46000 -> 64320 rows after gap-filling
Remaining nulls: 0


## Save cleaned CSVs to data/processed/

In [4]:
fund_master.to_csv(PROCESSED_DIR / "01_fund_master_clean.csv", index=False)
nav_history_clean.to_csv(PROCESSED_DIR / "02_nav_history_clean.csv", index=False)
scheme_perf.to_csv(PROCESSED_DIR / "07_scheme_performance_clean.csv", index=False)
transactions.to_csv(PROCESSED_DIR / "08_investor_transactions_clean.csv", index=False)
holdings.to_csv(PROCESSED_DIR / "09_portfolio_holdings_clean.csv", index=False)
benchmark.to_csv(PROCESSED_DIR / "10_benchmark_indices_clean.csv", index=False)

print("Cleaned CSVs written to", PROCESSED_DIR)

Cleaned CSVs written to ../data/processed


## Load into SQLite
Writes each cleaned table into `data/db/bluestock_mf.db`, matching `sql/schema.sql`.

In [5]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

fund_master.to_sql("fund_master", conn, if_exists="replace", index=False)
nav_history_clean.to_sql("nav_history", conn, if_exists="replace", index=False)
scheme_perf.to_sql("scheme_performance", conn, if_exists="replace", index=False)
transactions.to_sql("investor_transactions", conn, if_exists="replace", index=False)
holdings.to_sql("portfolio_holdings", conn, if_exists="replace", index=False)
benchmark.to_sql("benchmark_indices", conn, if_exists="replace", index=False)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

conn.close()

Tables in database:
                    name
0            fund_master
1            nav_history
2     scheme_performance
3  investor_transactions
4     portfolio_holdings
5      benchmark_indices
